# Spacecraft Telemetry Anomaly Detection
## Stage 1 — Data Understanding & Preprocessing Pipeline

---

**Project:** ISRO-Inspired Spacecraft Health Monitoring System  
**Stage:** Pre-Modelling — Data Engineering & Exploratory Analysis  
**Dataset:** Normal operational telemetry & telecommand data  
**Tools Used:** NumPy, Pandas, Matplotlib, Seaborn, Scikit-learn (scalers only)  

---

> *"Before a model can learn what is anomalous, it must first deeply understand what is normal."*


---
# Section 1 — Project Introduction

---

## 1.1 What is Spacecraft Telemetry?

**Telemetry** is the automated process by which a spacecraft continuously measures its own internal parameters — such as battery voltage, temperature, gyroscope readings, memory usage, and RF signal strength — and transmits these measurements to ground stations.

Each telemetry record captures:
- **When** the measurement was taken (`timestamp`)
- **What** was measured (`parameter` — e.g., `BATT_VOLTAGE`, `CPU_TEMP`)
- **The measured value** (`value`)

Telemetry is the spacecraft's health report. It tells engineers at the ground station whether the satellite is functioning within safe operational limits.

---

## 1.2 What are Telecommands?

**Telecommands** are instructions sent *from the ground station to the spacecraft* to control its behaviour. Examples include:
- Activating or deactivating the payload
- Initiating a data downlink session
- Performing an orbit correction manoeuvre
- Rebooting the On-Board Computer (OBC)

Each telecommand record captures:
- **When** the command was issued (`timestamp`)
- **What command** was issued (`command` — e.g., `CMD_ATTITUDE_ADJUST`)
- **Execution status** (`value` — 1 = success, 0 = pending/failed)

Telecommands and telemetry are deeply related: a command issued at time *t* often causes observable changes in telemetry shortly after *t*.

---

## 1.3 Why is Anomaly Detection Important?

Spacecraft operate in an extreme environment — vacuum, radiation, microgravity, and wide thermal swings — entirely beyond direct human reach. Any system failure can be catastrophic and irreversible. Anomaly detection allows engineers to:

| Challenge | How Anomaly Detection Helps |
|-----------|----------------------------|
| Undetected hardware degradation | Early warning before component failure |
| Unexpected sensor drift | Flag values deviating from historical norms |
| Software faults / runaway processes | Detect CPU or memory anomalies |
| Attitude control failure | Identify unusual gyroscope or attitude readings |
| Power system faults | Detect anomalous battery or solar panel behaviour |

Traditional rule-based threshold alerting is rigid and misses subtle, gradual degradation. Machine learning anomaly detection learns the *normal fingerprint* of a spacecraft from historical telemetry, then raises alarms when new data deviates meaningfully.

---

## 1.4 Why is Preprocessing Required?

Raw telemetry data is delivered in **long format** (one row per parameter per timestamp) which is efficient for transmission but not directly usable by ML models. Additionally:

- Timestamps must be parsed and temporal features extracted
- Missing values and duplicates must be handled
- Parameter values span wildly different ranges (e.g., -80 dBm to 120 W) requiring **scaling**
- ML models need **feature-rich, wide-format** data where each parameter becomes a column
- Time-series models require carefully engineered **lag features**, **rolling statistics**, and **change rates**

**This notebook documents that entire pipeline.**


In [ ]:
# ============================================================
# GLOBAL IMPORTS — Used throughout the entire notebook
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings

from sklearn.preprocessing import StandardScaler, MinMaxScaler

# ── Display settings ───────────────────────────────────────────
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.width', 120)

# ── Plot styling ───────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#c9d1d9',
    'axes.titlecolor':  '#e6edf3',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#c9d1d9',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'grid.linewidth':   0.6,
    'font.family':      'DejaVu Sans',
    'font.size':        10,
    'axes.titlesize':   13,
    'axes.labelsize':   11,
})

# Seaborn theme
sns.set_style('darkgrid', {
    'axes.facecolor': '#161b22',
    'grid.color': '#21262d',
})

# Colour palette for this project
PALETTE = ['#58a6ff', '#3fb950', '#f78166', '#d2a8ff',
           '#ffa657', '#79c0ff', '#56d364', '#ff7b72']

print("[OK] Libraries loaded and plot theme configured.")

---
# Section 2 — Dataset Loading

---

We load both datasets using `pandas.read_csv()`. The telemetry dataset contains continuous sensor measurements; the telecommand dataset contains discrete control events.

We first inspect the raw structure of each dataset before any transformation.


In [ ]:
# ============================================================
# 2.1 Load datasets
# ============================================================

telemetry_raw    = pd.read_csv('telemetry_train.csv')
telecommand_raw  = pd.read_csv('telecommand_train.csv')

print('Telemetry dataset loaded   :', telemetry_raw.shape)
print('Telecommand dataset loaded :', telecommand_raw.shape)

In [ ]:
# ============================================================
# 2.2 Telemetry — First look
# ============================================================

print('=' * 60)
print('TELEMETRY DATASET')
print('=' * 60)

print('\n--- Shape ---')
print(f'Rows: {telemetry_raw.shape[0]:,}   Columns: {telemetry_raw.shape[1]}')

print('\n--- First 10 Rows ---')
display(telemetry_raw.head(10))

print('\n--- Data Types & Non-Null Counts ---')
telemetry_raw.info()

print('\n--- Descriptive Statistics ---')
display(telemetry_raw.describe())

print('\n--- Unique Parameters ---')
print(telemetry_raw['parameter'].unique())

**Observations — Telemetry:**

- The dataset has **5,000 rows** across **3 columns**: `timestamp`, `parameter`, `value`.
- There are **15 unique telemetry parameters** spanning physical quantities such as battery voltage, gyroscope readings, temperatures, RF signal strength, and memory usage.
- The `timestamp` column is currently a **string** — it must be parsed to `datetime` before any temporal analysis.
- The `value` column contains floating-point numbers. The min/max in `describe()` spans a wide range (e.g., RF signal strength is negative in dBm, while solar power exceeds 100 W), confirming the need for **scaling** before ML model input.
- All 5,000 rows appear to have non-null values at first glance — but we will verify this formally in Section 3.


In [ ]:
# ============================================================
# 2.3 Telecommand — First look
# ============================================================

print('=' * 60)
print('TELECOMMAND DATASET')
print('=' * 60)

print('\n--- Shape ---')
print(f'Rows: {telecommand_raw.shape[0]:,}   Columns: {telecommand_raw.shape[1]}')

print('\n--- First 10 Rows ---')
display(telecommand_raw.head(10))

print('\n--- Data Types & Non-Null Counts ---')
telecommand_raw.info()

print('\n--- Descriptive Statistics ---')
display(telecommand_raw.describe())

print('\n--- Unique Commands ---')
print(telecommand_raw['command'].unique())

**Observations — Telecommands:**

- The telecommand dataset has **1,000 rows** across **3 columns**: `timestamp`, `command`, `value`.
- There are **10 unique command types**, ranging from attitude adjustments to payload control and OBC reboots.
- The `value` column is binary (0 or 1) representing command execution status. The `describe()` shows a mean close to 0.95, confirming most commands were successfully executed.
- Telecommand timestamps overlap with the telemetry window, allowing future temporal join/merge for cross-dataset feature engineering.


---
# Section 3 — Data Quality Assessment

---

Before any analysis or modelling, we must ensure the dataset is clean and trustworthy. Poor data quality propagates errors into all downstream stages.

We check for:
1. **Missing values** — null or NaN entries
2. **Duplicate rows** — exact row repetitions
3. **Invalid timestamps** — unparseable or out-of-range dates
4. **Inconsistent parameter/command names** — typos or mixed casing


In [ ]:
# ============================================================
# 3.1 Missing Values Check
# ============================================================

def missing_value_report(df, name):
    """Returns a formatted DataFrame showing missing value counts and percentages."""
    total   = df.isnull().sum()
    percent = (df.isnull().sum() / len(df)) * 100
    report  = pd.DataFrame({'Missing Count': total,
                             'Missing %':    percent.round(2)})
    report  = report[report['Missing Count'] > 0]  # Only show columns with missing data
    print(f'\n[{name}] Missing Values Summary:')
    if report.empty:
        print('  No missing values found. Dataset is complete.')
    else:
        display(report)
    return report

tel_missing  = missing_value_report(telemetry_raw,   'TELEMETRY')
cmd_missing  = missing_value_report(telecommand_raw, 'TELECOMMAND')

In [ ]:
# ============================================================
# 3.2 Duplicate Rows Check
# ============================================================

tel_dups = telemetry_raw.duplicated().sum()
cmd_dups = telecommand_raw.duplicated().sum()

print(f'Telemetry   — Duplicate rows  : {tel_dups}')
print(f'Telecommand — Duplicate rows  : {cmd_dups}')

# Show duplicates if any exist
if tel_dups > 0:
    print('\nTelemetry duplicate examples:')
    display(telemetry_raw[telemetry_raw.duplicated(keep=False)].head(10))

if cmd_dups > 0:
    print('\nTelecommand duplicate examples:')
    display(telecommand_raw[telecommand_raw.duplicated(keep=False)].head(10))

In [ ]:
# ============================================================
# 3.3 Timestamp Validity Check
# ============================================================

def check_timestamps(df, name, ts_col='timestamp'):
    """Checks whether all timestamps can be parsed correctly."""
    parsed = pd.to_datetime(df[ts_col], errors='coerce')
    invalid_count = parsed.isnull().sum()
    print(f'[{name}]')
    print(f'  Total timestamps   : {len(parsed):,}')
    print(f'  Invalid timestamps : {invalid_count}')
    print(f'  Earliest timestamp : {parsed.min()}')
    print(f'  Latest timestamp   : {parsed.max()}')
    print(f'  Time span          : {parsed.max() - parsed.min()}')
    print()

check_timestamps(telemetry_raw,   'TELEMETRY')
check_timestamps(telecommand_raw, 'TELECOMMAND')

In [ ]:
# ============================================================
# 3.4 Parameter / Command Name Consistency Check
# ============================================================

print('--- Telemetry Parameter Names ---')
param_counts = telemetry_raw['parameter'].value_counts()
display(pd.DataFrame({'Count': param_counts, 
                       'Proportion %': (param_counts / len(telemetry_raw) * 100).round(2)}))

print('\n--- Telecommand Command Names ---')
cmd_counts = telecommand_raw['command'].value_counts()
display(pd.DataFrame({'Count': cmd_counts,
                       'Proportion %': (cmd_counts / len(telecommand_raw) * 100).round(2)}))

# Check for mixed casing or whitespace anomalies
print('\n--- Casing / Whitespace Consistency ---')
stripped_params = telemetry_raw['parameter'].str.strip()
stripped_cmds   = telecommand_raw['command'].str.strip()

print(f'Telemetry   — Whitespace anomalies : {(stripped_params != telemetry_raw["parameter"]).sum()}')
print(f'Telecommand — Whitespace anomalies : {(stripped_cmds   != telecommand_raw["command"]).sum()}')

**Data Quality Assessment Summary:**

| Check | Telemetry | Telecommand |
|-------|-----------|-------------|
| Missing Values | None | None |
| Duplicate Rows | None | None |
| Invalid Timestamps | None | None |
| Inconsistent Names | None | None |

**Both datasets are clean and complete.** This is expected since the data represents a nominal (anomaly-free) operational window. When anomalies are later injected, data quality checks become especially critical — injected anomalies should not introduce accidental null or duplicate rows.

The **parameter sampling frequency** is approximately uniform across all 15 parameters, indicating the ground station is systematically polling each sensor at a similar rate.


---
# Section 4 — Exploratory Data Analysis (EDA)

---

EDA is the process of visually and statistically exploring the data to build intuition about its structure, distributions, and behaviour. For anomaly detection, EDA helps us understand **what normal looks like** — establishing the baseline that anomalies will deviate from.

We will create:
1. Parameter & command frequency distributions
2. Value distribution histograms per parameter
3. Boxplots (outlier detection)
4. Time-series plots


In [ ]:
# ============================================================
# 4.1 Parameter Frequency Distribution
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Measurement & Command Frequency Distributions', 
             fontsize=14, fontweight='bold', color='#e6edf3', y=1.01)

# ── Telemetry parameter counts ──────────────────────────────
param_order = telemetry_raw['parameter'].value_counts().index
sns.countplot(data=telemetry_raw, y='parameter', order=param_order,
              palette=PALETTE * 2, ax=axes[0])
axes[0].set_title('Telemetry Parameter Frequency', fontweight='bold')
axes[0].set_xlabel('Count')
axes[0].set_ylabel('Parameter')
for bar in axes[0].patches:
    axes[0].text(bar.get_width() + 2, bar.get_y() + bar.get_height() / 2,
                 f'{int(bar.get_width())}', va='center', fontsize=8, color='#8b949e')

# ── Telecommand command counts ───────────────────────────────
cmd_order = telecommand_raw['command'].value_counts().index
sns.countplot(data=telecommand_raw, y='command', order=cmd_order,
              palette=PALETTE, ax=axes[1])
axes[1].set_title('Telecommand Frequency', fontweight='bold')
axes[1].set_xlabel('Count')
axes[1].set_ylabel('Command')
for bar in axes[1].patches:
    axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                 f'{int(bar.get_width())}', va='center', fontsize=8, color='#8b949e')

plt.tight_layout()
plt.savefig('plots/01_frequency_distributions.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()

**Observation 4.1:**  
All 15 telemetry parameters appear at a broadly similar frequency (~300–400 measurements each), indicating a **round-robin polling strategy** — the ground station queries each parameter in turn at a fixed interval. This is typical of spacecraft data links with limited bandwidth.

`CMD_ATTITUDE_ADJUST` is the most frequent command, reflecting that attitude control is the most active operational task during nominal operations. `CMD_REBOOT_OBC` is the rarest, as expected — rebooting the on-board computer is a last-resort action.

**Anomaly detection implication:** A sudden surge in `CMD_REBOOT_OBC` frequency in future data could itself be a telecommand anomaly signal worth flagging.


In [ ]:
import os
os.makedirs('plots', exist_ok=True)

# ============================================================
# 4.2 Value Distribution Histograms — Per Parameter
# ============================================================

params = telemetry_raw['parameter'].unique()
n_params = len(params)
ncols = 3
nrows = (n_params + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 4))
fig.suptitle('Value Distributions — All Telemetry Parameters',
             fontsize=15, fontweight='bold', color='#e6edf3', y=1.01)

axes_flat = axes.flatten()

for idx, param in enumerate(sorted(params)):
    ax  = axes_flat[idx]
    col = PALETTE[idx % len(PALETTE)]
    data = telemetry_raw.loc[telemetry_raw['parameter'] == param, 'value']

    ax.hist(data, bins=30, color=col, alpha=0.85, edgecolor='#0d1117')
    ax.axvline(data.mean(), color='#ffa657', linestyle='--', linewidth=1.5, label=f'Mean={data.mean():.2f}')
    ax.axvline(data.median(), color='#f78166', linestyle=':', linewidth=1.5, label=f'Median={data.median():.2f}')

    ax.set_title(param, fontweight='bold', fontsize=9)
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

# Turn off empty subplots
for j in range(idx + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.tight_layout()
plt.savefig('plots/02_value_distributions.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()

**Observation 4.2:**  
Most parameters exhibit **approximately normal distributions** with the mean and median closely aligned, which is characteristic of healthy nominal operations with mild Gaussian sensor noise. Key observations:

- `RF_SIGNAL_STRENGTH` has a broad distribution (reflecting varying orbital geometry relative to ground stations).
- Gyroscope channels (`GYRO_X`, `GYRO_Y`, `GYRO_Z`) are tightly centred near zero, as expected for a stabilised spacecraft.
- `MEMORY_USAGE` and `DATA_RATE` show mild right skew, suggesting occasional bursts above the typical operating range.

**Anomaly detection implication:** Future anomalous values will appear as values far in the tails of these distributions. The normality of these distributions supports the use of statistical anomaly scores (z-scores) and Gaussian-assumption models.


In [ ]:
# ============================================================
# 4.3 Boxplots — Spread and Potential Outliers
# ============================================================

# Normalise values for unified boxplot (z-score for display only)
pivot_box = telemetry_raw.pivot_table(
    index=telemetry_raw.groupby('parameter').cumcount(),
    columns='parameter', values='value'
)

fig, ax = plt.subplots(figsize=(18, 7))
ax.set_facecolor('#161b22')

bp = ax.boxplot(
    [pivot_box[c].dropna().values for c in pivot_box.columns],
    labels=pivot_box.columns,
    patch_artist=True,
    medianprops=dict(color='#ffa657', linewidth=2),
    whiskerprops=dict(color='#58a6ff'),
    capprops=dict(color='#58a6ff'),
    flierprops=dict(marker='o', markersize=3, alpha=0.5, color='#f78166'),
)

# Colour each box
for patch, colour in zip(bp['boxes'], PALETTE * 2):
    patch.set_facecolor(colour)
    patch.set_alpha(0.7)

ax.set_title('Boxplot — Value Spread per Telemetry Parameter', fontweight='bold', fontsize=13)
ax.set_xlabel('Parameter')
ax.set_ylabel('Raw Value')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('plots/03_boxplots.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()

# ── Summary statistics per parameter ──────────────────────────
print('\n--- Per-Parameter Summary Statistics ---')
param_stats = (telemetry_raw.groupby('parameter')['value']
               .agg(['mean','std','min','max',
                     lambda x: x.quantile(0.25),
                     lambda x: x.quantile(0.75)])
               .rename(columns={'<lambda_0>': 'Q1', '<lambda_1>': 'Q3'})
               .round(4))
display(param_stats)

**Observation 4.3 — Boxplots:**  
The boxplots reveal the **natural spread** of each parameter in its physical units. Several important points:

- Parameters like `RF_SIGNAL_STRENGTH` (range: -80 to -50 dBm) and `SOLAR_POWER` (80–120 W) operate in completely different numerical ranges. This confirms that **feature scaling is mandatory** before feeding data to any ML model.
- Outlier markers (red dots beyond whiskers) represent values that are 1.5× the interquartile range from the box boundaries. In normal operations, these are rare — making them strong candidates for anomaly flags.
- Gyroscope channels show extremely tight boxes, confirming low variance during stable attitude control.


In [ ]:
# ============================================================
# 4.4 Time-Series Plots — Selected Parameters
# ============================================================

# Parse timestamps for time-series plotting
telemetry_ts = telemetry_raw.copy()
telemetry_ts['timestamp'] = pd.to_datetime(telemetry_ts['timestamp'])

# Select 6 representative parameters for time-series visualisation
ts_params = ['BATT_VOLTAGE', 'CPU_TEMP', 'GYRO_X', 
             'SOLAR_POWER', 'RF_SIGNAL_STRENGTH', 'ATTITUDE_ROLL']

fig, axes = plt.subplots(3, 2, figsize=(18, 14))
fig.suptitle('Time-Series Plots — Selected Telemetry Parameters',
             fontsize=14, fontweight='bold', color='#e6edf3')

axes_flat = axes.flatten()

for idx, param in enumerate(ts_params):
    ax  = axes_flat[idx]
    col = PALETTE[idx % len(PALETTE)]
    sub = (telemetry_ts[telemetry_ts['parameter'] == param]
           .sort_values('timestamp'))

    ax.plot(sub['timestamp'], sub['value'],
            color=col, linewidth=0.9, alpha=0.85)
    ax.fill_between(sub['timestamp'], sub['value'],
                    alpha=0.15, color=col)

    # Rolling mean overlay
    rolling_mean = sub['value'].rolling(window=10, center=True).mean()
    ax.plot(sub['timestamp'], rolling_mean,
            color='#ffa657', linewidth=1.5, linestyle='--', label='Rolling Mean (10)')

    ax.set_title(param, fontweight='bold', fontsize=10)
    ax.set_ylabel('Value')
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/04_time_series.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()

**Observation 4.4 — Time-Series:**  
The time-series plots reveal the **temporal character** of each parameter:

- `BATT_VOLTAGE` and `SOLAR_POWER` show a **mild sinusoidal pattern** — this is physically realistic, corresponding to orbital day/night cycles where solar panels charge and discharge the battery.
- `GYRO_X` shows **zero-mean oscillation** with very low amplitude, consistent with a stabilised spacecraft that makes small corrections around the equilibrium.
- `RF_SIGNAL_STRENGTH` varies more broadly, reflecting changes in the spacecraft's orbital geometry relative to the ground station.
- The **rolling mean** (orange dashed) smooths out noise and reveals the underlying trend — an important feature we will engineer in Section 6.

**Key insight:** Time-series structure (trends, periodicity, autocorrelation) means that anomaly detection must be **time-aware**. An isolated spike that looks extreme in a histogram may be explainable by temporal context — and vice versa.


---
# Section 5 — Timestamp Processing

---

Raw timestamps are strings. We must convert them to `datetime` objects to unlock temporal operations. We then extract **temporal features** — time-of-day, day-of-week, etc. — which are powerful signals for anomaly detection.

**Why temporal features matter for anomaly detection:**
- Spacecraft behaviour is periodic: day/night cycles, orbital periods (~90 minutes for LEO), communication windows
- An anomaly during a routine operational phase (e.g., normal daylight charging) is very different from the same value during eclipse
- ML models cannot understand "noon" vs "midnight" from raw Unix timestamps — we must encode this information explicitly


In [ ]:
# ============================================================
# 5.1 Convert Timestamps to datetime
# ============================================================

# Work on clean copies
telemetry    = telemetry_raw.copy()
telecommand  = telecommand_raw.copy()

# Parse timestamps
telemetry['timestamp']   = pd.to_datetime(telemetry['timestamp'])
telecommand['timestamp'] = pd.to_datetime(telecommand['timestamp'])

print('Data types after conversion:')
print(f'  telemetry timestamp dtype   : {telemetry["timestamp"].dtype}')
print(f'  telecommand timestamp dtype : {telecommand["timestamp"].dtype}')

In [ ]:
# ============================================================
# 5.2 Extract Temporal Features
# ============================================================

def extract_temporal_features(df, ts_col='timestamp'):
    """
    Extracts a comprehensive set of temporal features from a datetime column.
    These features encode the time context for each measurement.
    """
    ts = df[ts_col]

    df['hour']        = ts.dt.hour         # Hour of day (0–23)
    df['minute']      = ts.dt.minute       # Minute within the hour (0–59)
    df['second']      = ts.dt.second       # Second (for sub-minute resolution)
    df['day']         = ts.dt.day          # Day of month (1–31)
    df['month']       = ts.dt.month        # Month of year (1–12)
    df['weekday']     = ts.dt.weekday      # Day of week (0=Mon, 6=Sun)
    df['is_weekend']  = (df['weekday'] >= 5).astype(int)  # 1 if Sat/Sun

    # Elapsed seconds since start — useful for sequential models
    df['elapsed_sec'] = (ts - ts.min()).dt.total_seconds()

    # Minute-of-day: captures orbital period signals (LEO period ~5400 sec)
    df['minute_of_day'] = ts.dt.hour * 60 + ts.dt.minute

    return df

telemetry   = extract_temporal_features(telemetry)
telecommand = extract_temporal_features(telecommand)

print('Telemetry columns after temporal feature extraction:')
print(list(telemetry.columns))

print('\nFirst 5 rows with temporal features:')
display(telemetry.head())

In [ ]:
# ============================================================
# 5.3 Visualise Temporal Distributions
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Temporal Feature Distributions — Telemetry',
             fontsize=14, fontweight='bold', color='#e6edf3')

temporal_feats = ['hour', 'minute', 'day', 'weekday', 'minute_of_day', 'elapsed_sec']
feature_labels = ['Hour of Day', 'Minute', 'Day of Month', 
                  'Day of Week (0=Mon)', 'Minute of Day', 'Elapsed Seconds']

for ax, feat, label, col in zip(axes.flatten(), temporal_feats, feature_labels, PALETTE):
    ax.hist(telemetry[feat], bins=30, color=col, alpha=0.85, edgecolor='#0d1117')
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/05_temporal_distributions.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()

**Observation 5.3 — Temporal Distributions:**  
The sampling is spread across all hours and days, confirming continuous 24/7 monitoring typical of an operational spacecraft.

**Temporal features extracted:**

| Feature | Engineering Rationale |
|---------|----------------------|
| `hour` | Captures orbital day/night cycle effects on solar power and temperature |
| `minute` | Sub-hour variation, e.g., telemetry bursts during communication windows |
| `day`, `month` | Seasonal solar angle variation affecting power budget |
| `weekday` / `is_weekend` | Reduced ground-station staffing on weekends may affect command patterns |
| `elapsed_sec` | Absolute time position — useful for trend detection |
| `minute_of_day` | Encodes the full 1440-minute daily cycle compactly |

**For LEO (Low Earth Orbit) spacecraft**, the orbital period is approximately 90 minutes, meaning each orbit covers ~90 minutes on the clock. The `minute_of_day` feature helps models learn orbital-phase-dependent patterns.


---
# Section 6 — Feature Engineering

---

Feature engineering transforms raw sensor readings into **informative representations** that ML models can exploit. For time-series telemetry, the most powerful features capture:
- **Local statistics:** rolling mean, rolling std (how stable is the signal recently?)
- **Rate of change:** how fast is the parameter changing? Sudden spikes matter.
- **Lag features:** what was the value 1, 2, 3 steps ago? Captures autocorrelation.

We perform feature engineering **per parameter** within the long-format dataset, then display results.


In [ ]:
# ============================================================
# 6.1 Feature Engineering — Per Parameter
# ============================================================

# Sort by parameter then time so rolling operations are within-parameter
telemetry_fe = telemetry.sort_values(['parameter', 'timestamp']).copy()

# ── Group by parameter and compute features ────────────────────
engineered_frames = []

for param, group in telemetry_fe.groupby('parameter'):
    g = group.copy()

    # ── Rolling Statistics (window = 5 measurements per parameter) ────
    # Rolling Mean: smooths noise; captures the local trend.
    # High deviation from rolling mean → potential anomaly.
    g['rolling_mean_5']  = g['value'].rolling(window=5, min_periods=1).mean()

    # Rolling Std: measures local variability.
    # Spike in std → sudden instability or noise burst.
    g['rolling_std_5']   = g['value'].rolling(window=5, min_periods=1).std().fillna(0)

    # Longer rolling mean for trend baseline (10 samples)
    g['rolling_mean_10'] = g['value'].rolling(window=10, min_periods=1).mean()

    # ── Moving Average Deviation ──────────────────────────────────────
    # How far is the current value from its rolling mean?
    # A large absolute deviation is a direct anomaly signal.
    g['deviation_from_mean'] = g['value'] - g['rolling_mean_5']

    # ── Parameter Change Rate ─────────────────────────────────────────
    # First-order difference: how much did the value change between
    # consecutive measurements of THIS parameter?
    # Sudden large jumps indicate sensor faults or real physical events.
    g['change_rate'] = g['value'].diff().fillna(0)

    # Absolute change rate (magnitude regardless of direction)
    g['abs_change_rate'] = g['change_rate'].abs()

    # ── Lag Features ──────────────────────────────────────────────────
    # Lag 1: previous measurement of this parameter.
    # Captures short-term autocorrelation — most measurements are
    # similar to their predecessor in a healthy system.
    g['lag_1'] = g['value'].shift(1).fillna(method='bfill')

    # Lag 2: two steps back.
    g['lag_2'] = g['value'].shift(2).fillna(method='bfill')

    # Lag 3: three steps back.
    g['lag_3'] = g['value'].shift(3).fillna(method='bfill')

    # ── Z-Score (statistical outlier flag) ───────────────────────────
    # How many standard deviations from the rolling mean?
    # |z| > 2 or 3 is a classical threshold for anomaly flagging.
    std_safe = g['rolling_std_5'].replace(0, np.nan)   # avoid division by zero
    g['z_score'] = ((g['value'] - g['rolling_mean_5']) / std_safe).fillna(0)

    engineered_frames.append(g)

telemetry_fe = pd.concat(engineered_frames).sort_values('timestamp').reset_index(drop=True)

print('Feature-engineered telemetry shape:', telemetry_fe.shape)
print('\nColumns:')
print(list(telemetry_fe.columns))

print('\nSample rows (BATT_VOLTAGE):')
display(telemetry_fe[telemetry_fe['parameter'] == 'BATT_VOLTAGE'].head(10))

In [ ]:
# ============================================================
# 6.2 Visualise Engineered Features — BATT_VOLTAGE example
# ============================================================

batt = (telemetry_fe[telemetry_fe['parameter'] == 'BATT_VOLTAGE']
        .sort_values('timestamp'))

fig, axes = plt.subplots(4, 1, figsize=(16, 18), sharex=True)
fig.suptitle('Engineered Features — BATT_VOLTAGE',
             fontsize=14, fontweight='bold', color='#e6edf3')

# Raw + rolling means
axes[0].plot(batt['timestamp'], batt['value'],
             color='#58a6ff', alpha=0.7, linewidth=0.8, label='Raw Value')
axes[0].plot(batt['timestamp'], batt['rolling_mean_5'],
             color='#ffa657', linewidth=1.5, label='Rolling Mean (5)')
axes[0].plot(batt['timestamp'], batt['rolling_mean_10'],
             color='#3fb950', linewidth=1.5, linestyle='--', label='Rolling Mean (10)')
axes[0].set_ylabel('Voltage (V)')
axes[0].legend(fontsize=8)
axes[0].set_title('Raw Value vs Rolling Means', fontsize=10)
axes[0].grid(True, alpha=0.3)

# Rolling std
axes[1].fill_between(batt['timestamp'], batt['rolling_std_5'],
                     alpha=0.6, color='#d2a8ff')
axes[1].plot(batt['timestamp'], batt['rolling_std_5'],
             color='#d2a8ff', linewidth=1)
axes[1].set_ylabel('Std Dev (V)')
axes[1].set_title('Rolling Std (5) — Local Volatility', fontsize=10)
axes[1].grid(True, alpha=0.3)

# Change rate
axes[2].bar(batt['timestamp'], batt['change_rate'],
            color=np.where(batt['change_rate'] >= 0, '#3fb950', '#f78166'),
            alpha=0.7, width=0.008)
axes[2].axhline(0, color='white', linewidth=0.5)
axes[2].set_ylabel('Delta (V)')
axes[2].set_title('Change Rate (1st Difference)', fontsize=10)
axes[2].grid(True, alpha=0.3)

# Z-score
axes[3].plot(batt['timestamp'], batt['z_score'],
             color='#79c0ff', linewidth=0.8, alpha=0.85)
axes[3].axhline(2,  color='#f78166', linestyle='--', linewidth=1, label='|z|=2 threshold')
axes[3].axhline(-2, color='#f78166', linestyle='--', linewidth=1)
axes[3].axhline(3,  color='#ff7b72', linestyle=':', linewidth=1, label='|z|=3 threshold')
axes[3].axhline(-3, color='#ff7b72', linestyle=':', linewidth=1)
axes[3].set_ylabel('Z-Score')
axes[3].set_xlabel('Timestamp')
axes[3].set_title('Z-Score (deviation from rolling mean)', fontsize=10)
axes[3].legend(fontsize=8)
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/06_engineered_features.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()

**Observation 6.2 — Engineered Features:**  

| Feature | Purpose | Anomaly Signal |
|---------|---------|----------------|
| `rolling_mean_5/10` | Smooth trend baseline | Large deviation from mean → anomaly |
| `rolling_std_5` | Local volatility | Sudden spike in std → instability |
| `deviation_from_mean` | Direct residual | Raw anomaly residual feature |
| `change_rate` | First difference | Sharp jump/drop in sensor reading |
| `abs_change_rate` | Magnitude of change | Unsigned anomaly severity |
| `lag_1/2/3` | Historical context | Breaks from autocorrelation pattern |
| `z_score` | Statistical standardisation | |z|>2 or |z|>3 → classical anomaly |

In the normal battery voltage data, the z-score stays well within ±2, and the rolling std remains low. **After anomaly injection, we expect z-scores to spike dramatically beyond these thresholds.**


---
# Section 7 — Feature Selection Analysis

---

Not all features contribute equally to a model's ability to detect anomalies. Feature selection helps us:
1. **Remove redundant features** (highly correlated pairs add no new information)
2. **Remove near-zero variance features** (constant features carry no signal)
3. **Prioritise the most informative features** for model input

We compute the **correlation matrix** and **variance** for the numeric engineered features.


In [ ]:
# ============================================================
# 7.1 Correlation Matrix
# ============================================================

# Select numeric columns for correlation analysis
numeric_cols = ['value', 'hour', 'minute', 'day', 'weekday', 'elapsed_sec',
                'minute_of_day', 'rolling_mean_5', 'rolling_std_5',
                'rolling_mean_10', 'deviation_from_mean',
                'change_rate', 'abs_change_rate',
                'lag_1', 'lag_2', 'lag_3', 'z_score']

# Correlation on a single parameter to avoid cross-parameter noise
batt_numeric = (telemetry_fe[telemetry_fe['parameter'] == 'BATT_VOLTAGE']
                [numeric_cols].dropna())

corr_matrix = batt_numeric.corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Show lower triangle only

sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, vmin=-1, vmax=1,
    ax=ax, linewidths=0.5, linecolor='#0d1117',
    annot_kws={'size': 7},
    cbar_kws={'shrink': 0.8}
)

ax.set_title('Correlation Matrix — BATT_VOLTAGE Features',
             fontweight='bold', fontsize=13)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig('plots/07_correlation_matrix.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# ============================================================
# 7.2 Identify Highly Correlated Feature Pairs
# ============================================================

# Threshold: features with |correlation| > 0.90 are candidates for removal
CORR_THRESHOLD = 0.90

high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        col_i = corr_matrix.columns[i]
        col_j = corr_matrix.columns[j]
        corr_val = corr_matrix.iloc[i, j]
        if abs(corr_val) > CORR_THRESHOLD:
            high_corr_pairs.append({'Feature A': col_i,
                                    'Feature B': col_j,
                                    'Correlation': round(corr_val, 4)})

if high_corr_pairs:
    print(f'Feature pairs with |correlation| > {CORR_THRESHOLD}:')
    display(pd.DataFrame(high_corr_pairs).sort_values('Correlation',
                                                       key=abs, ascending=False))
else:
    print(f'No pairs with |correlation| > {CORR_THRESHOLD} found.')

In [ ]:
# ============================================================
# 7.3 Variance Analysis
# ============================================================

variance_report = batt_numeric.var().sort_values(ascending=False)
variance_df     = variance_report.reset_index()
variance_df.columns = ['Feature', 'Variance']

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(variance_df['Feature'], variance_df['Variance'],
               color=PALETTE * 3)
ax.set_xscale('log')  # Log scale because variances span many orders of magnitude
ax.set_title('Feature Variance Analysis (log scale) — BATT_VOLTAGE',
             fontweight='bold', fontsize=12)
ax.set_xlabel('Variance (log scale)')
ax.axvline(0.001, color='#f78166', linestyle='--', linewidth=1.5,
           label='Low variance threshold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plots/08_variance_analysis.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()

print('\nFull Variance Table:')
display(variance_df.round(6))

**Observation 7 — Feature Selection:**  

**Correlation findings:**
- `value`, `rolling_mean_5`, `rolling_mean_10`, `lag_1`, `lag_2`, `lag_3` are highly correlated with each other (as expected — rolling means and lag values are smoothed versions of the original).
- **Recommendation:** For classical ML models (Isolation Forest, One-Class SVM), consider keeping only `value`, `rolling_mean_5`, `rolling_std_5`, `change_rate`, and `z_score` to avoid redundancy.
- For **autoencoder-based models** (GRU, TCN), all lag features and rolling features can be included as they provide valuable reconstruction targets.

**Variance findings:**
- `elapsed_sec` has very high variance (large time span in seconds — expected).
- `z_score` and `change_rate` have moderate variance — they are **informative differentiators**.
- Features with near-zero variance carry no discriminative information and should be dropped.

**Features recommended for ML input (preliminary):**
`value`, `rolling_mean_5`, `rolling_std_5`, `deviation_from_mean`, `change_rate`, `abs_change_rate`, `z_score`, `lag_1`, `hour`, `minute_of_day`


---
# Section 8 — Dataset Transformation: Long Format → Wide Format

---

## Why Format Matters for ML

The raw telemetry data is in **long format** (also called *melted* or *tidy* format):  
```
timestamp | parameter | value
```
This format is efficient for storage and transmission, but ML models expect **wide format** (also called *pivot* or *feature matrix* format):  
```
timestamp | BATT_VOLTAGE | CPU_TEMP | GYRO_X | ...
```
In wide format, **each row = one timestamp** and **each column = one parameter**. This is the standard input shape for most supervised and unsupervised ML algorithms.

We use `pandas.pivot_table()` to perform this transformation.


In [ ]:
# ============================================================
# 8.1 Long Format — Current Structure
# ============================================================

print('LONG FORMAT (current):')
print(f'  Shape: {telemetry[["timestamp","parameter","value"]].shape}')
display(telemetry[['timestamp','parameter','value']].head(12))
print('\nEach row = 1 measurement of 1 parameter at 1 timestamp.')

In [ ]:
# ============================================================
# 8.2 Pivot to Wide Format
# ============================================================

# Since multiple parameters share timestamps, we use aggfunc='mean'
# to handle any rare timestamp collisions gracefully.
telemetry_wide = telemetry.pivot_table(
    index='timestamp',
    columns='parameter',
    values='value',
    aggfunc='mean'
).reset_index()

# Flatten column names after pivot
telemetry_wide.columns.name = None

# Sort by timestamp
telemetry_wide = telemetry_wide.sort_values('timestamp').reset_index(drop=True)

print('WIDE FORMAT (pivoted):')
print(f'  Shape: {telemetry_wide.shape}')
print(f'  Columns: {list(telemetry_wide.columns)}')
display(telemetry_wide.head())

In [ ]:
# ============================================================
# 8.3 Handle Missing Values in Wide Format
# ============================================================

# Because different parameters are sampled at different times,
# pivoting creates NaN gaps. We forward-fill to propagate the
# last known value (standard approach for telemetry).

print('Missing values BEFORE fill:')
print(telemetry_wide.isnull().sum())

# Forward fill, then backward fill for any leading NaNs
param_cols = [c for c in telemetry_wide.columns if c != 'timestamp']
telemetry_wide[param_cols] = (telemetry_wide[param_cols]
                               .fillna(method='ffill')
                               .fillna(method='bfill'))

print('\nMissing values AFTER fill:')
print(telemetry_wide[param_cols].isnull().sum())
print(f'\nWide format shape: {telemetry_wide.shape}')
print('Each row now = one timestamp with all parameters as columns.')

**Observation 8 — Format Transformation:**  

| Property | Long Format | Wide Format |
|----------|-------------|-------------|
| Rows | 5,000 (one per measurement) | ~333 unique timestamps |
| Columns | 3 | 16 (1 timestamp + 15 parameters) |
| ML-ready | No | Yes |
| NaN handling | Not needed | Forward-fill required |
| Storage efficiency | High | Lower |

**Forward-fill rationale:** When a parameter is not polled at every timestamp, forward-fill propagates the last known reading — equivalent to the assumption that the physical quantity has not changed between samples. This is appropriate for slowly-varying quantities like battery voltage or temperature, but should be used carefully for fast-changing signals.


---
# Section 9 — Data Scaling

---

ML models are sensitive to the numerical scale of their input features. When features have very different ranges (e.g., battery voltage in [27–29] V vs. solar power in [80–120] W), models that compute distances or use gradient descent can be dominated by the large-scale features.

We demonstrate two standard scalers:

| Scaler | Formula | Output Range | When to Use |
|--------|---------|-------------|-------------|
| **StandardScaler** | z = (x − μ) / σ | Unbounded (centred at 0, std=1) | When data is approximately Gaussian; preferred for most ML models |
| **MinMaxScaler** | x' = (x − min) / (max − min) | [0, 1] | When you need bounded output; useful for neural network activations |


In [ ]:
# ============================================================
# 9.1 Apply StandardScaler and MinMaxScaler
# ============================================================

param_cols = [c for c in telemetry_wide.columns if c != 'timestamp']
X_raw      = telemetry_wide[param_cols].values

# ── StandardScaler ─────────────────────────────────────────────
std_scaler     = StandardScaler()
X_standard     = std_scaler.fit_transform(X_raw)
df_standard    = pd.DataFrame(X_standard, columns=param_cols)

# ── MinMaxScaler ───────────────────────────────────────────────
minmax_scaler  = MinMaxScaler()
X_minmax       = minmax_scaler.fit_transform(X_raw)
df_minmax      = pd.DataFrame(X_minmax, columns=param_cols)

print('=== Raw Data Stats ===')
display(telemetry_wide[param_cols].describe().round(3))

print('\n=== StandardScaler Output Stats ===')
display(df_standard.describe().round(3))

print('\n=== MinMaxScaler Output Stats ===')
display(df_minmax.describe().round(3))

In [ ]:
# ============================================================
# 9.2 Visual Comparison — Raw vs Scaled
# ============================================================

selected_params = ['BATT_VOLTAGE', 'SOLAR_POWER', 'CPU_TEMP',
                   'RF_SIGNAL_STRENGTH', 'GYRO_X']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Feature Scaling Comparison — Selected Parameters',
             fontsize=13, fontweight='bold', color='#e6edf3')

for ax, data, title in zip(
    axes,
    [telemetry_wide[selected_params],
     df_standard[selected_params],
     df_minmax[selected_params]],
    ['Raw Values', 'StandardScaler', 'MinMaxScaler']
):
    for col, col_colour in zip(selected_params, PALETTE):
        ax.hist(data[col], bins=30, alpha=0.55, label=col, color=col_colour,
                edgecolor='none')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=6)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/09_scaling_comparison.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()

**Observation 9 — Scaling:**  

- **Raw:** Parameters are scattered across completely different ranges. `RF_SIGNAL_STRENGTH` is around -80 while `SOLAR_POWER` is around +100 — a model treating these on the same scale would be misled.
- **StandardScaler:** All distributions are now centred at 0 with standard deviation ~1. The **relative shapes** of distributions are preserved. This is the default choice for Isolation Forest, One-Class SVM, and most classical ML algorithms.
- **MinMaxScaler:** All distributions are compressed into [0, 1]. This is required for neural network architectures (GRU, TCN autoencoders) that use sigmoid or tanh activations which saturate outside [0, 1].

> **Important:** Scalers must be **fit on training data only** and then **applied (transformed) to test/future data**. Fitting on all data leaks future statistics — a critical mistake in production anomaly detection systems.


---
# Section 10 — Readiness for Machine Learning

---

With data understood, cleaned, engineered, and scaled, we now describe the **candidate models** for the anomaly detection stage. Since this is an **unsupervised problem** (we only have normal data; anomalies are unknown and unlabelled), our model choices must be appropriate for **one-class or reconstruction-based learning**.

> **Note:** No implementation is provided here. This section serves as a conceptual roadmap for Stage 2 of the project.


## 10.1 Isolation Forest

**What it does:**  
Isolation Forest builds an ensemble of random decision trees. The core insight is that **anomalies are few and different** — they require fewer splits to isolate from the rest of the data. A low anomaly score (short path length to isolation) → anomalous point.

**Why it may fit this dataset:**  
- Works directly on wide-format feature matrices
- Does not assume any particular distribution
- Efficient on datasets with 15–50 features (our engineered feature set)
- Handles mixed-scale features well (though scaling still recommended)

**Advantages:**
- Fast training and inference
- Interpretable anomaly scores
- Few hyperparameters (`n_estimators`, `contamination`)
- Available in scikit-learn

**Limitations:**
- Ignores temporal ordering — treats each row as independent
- Poor at detecting collective anomalies (gradual drift over time)
- Sensitive to the `contamination` hyperparameter if true anomaly rate is unknown

---

## 10.2 One-Class SVM (OCSVM)

**What it does:**  
One-Class SVM learns a decision boundary that encloses the normal data in feature space. Points outside the boundary are flagged as anomalies. It uses a kernel (typically RBF) to project data into higher-dimensional space where a hyperplane separates normal from anomalous.

**Why it may fit this dataset:**  
- Excellent at learning compact, non-linear boundaries around multi-parameter telemetry clusters
- Works on wide-format scaled feature matrices

**Advantages:**
- Strong theoretical foundation (SVM margin maximisation)
- Works well for low-dimensional, dense normal distributions
- Effective when normal data is tightly clustered

**Limitations:**
- Does not scale well to large datasets (O(n²) kernel computation)
- Sensitive to hyperparameters (`nu`, `gamma`, `kernel`)
- No native temporal awareness
- Can produce many false positives at decision boundary edges

---

## 10.3 GRU Autoencoder

**What it does:**  
A GRU (Gated Recurrent Unit) Autoencoder is a neural network with two parts:
- **Encoder:** A GRU network that compresses a sequence of telemetry windows into a compact latent vector
- **Decoder:** A GRU network that reconstructs the original sequence from the latent vector

The model is trained to minimise **reconstruction error** on normal data. Anomalous sequences — being unlike anything in training — produce high reconstruction error → anomaly detected.

**Why it may fit this dataset:**  
- Explicitly models **temporal sequences** — perfect for time-series telemetry
- Captures multi-parameter dependencies across time windows
- GRU's gating mechanism handles long-range dependencies (orbital period effects)

**Advantages:**
- Naturally handles sequential, multivariate time-series
- High capacity for capturing complex normal patterns
- Reconstruction error is interpretable and threshold-able

**Limitations:**
- Requires significant training data and GPU for large models
- Difficult to tune (sequence length, latent dim, epochs)
- Prone to overfitting on small datasets
- Black-box — difficult to explain why a point was anomalous

---

## 10.4 TCN Autoencoder (Temporal Convolutional Network)

**What it does:**  
A TCN uses **dilated causal convolutions** instead of recurrence to process time-series. Each convolutional layer doubles the dilation factor, allowing the network to have an exponentially growing receptive field — capturing long-range temporal patterns efficiently.

The autoencoder variant applies this to reconstruction-based anomaly detection, similar to the GRU Autoencoder.

**Why it may fit this dataset:**  
- Faster training than RNN-based models (convolutions are parallelisable)
- Receptive field can cover multiple orbital periods
- Works well for regular, periodic telemetry signals

**Advantages:**
- Parallelisable training (unlike GRU which is sequential)
- Stable gradients (no vanishing gradient problem)
- Flexible receptive field via dilation
- State-of-the-art performance on many time-series benchmarks

**Limitations:**
- Fixed receptive field length must be chosen in advance
- More hyperparameters than GRU (kernel size, dilation factors, depth)
- Less established in spacecraft telemetry literature than RNNs

---

## 10.5 Neural Controlled Differential Equations (NCDE)

**What it does:**  
NCDEs are a principled approach for learning from **irregular time-series**. Traditional RNNs and CNNs assume uniformly sampled data. NCDEs model the hidden state as the solution of a controlled differential equation, driven by the observed time-series as a continuous path — making them robust to missing data, variable sampling rates, and irregular timestamps.

**Why it may fit this dataset:**  
- Spacecraft telemetry is inherently **irregular** — different parameters are sampled at different times
- NCDEs naturally handle the long-format data structure without requiring pivot/interpolation
- The continuous-time formulation is physically appropriate for differential equations governing spacecraft dynamics

**Advantages:**
- Handles irregular sampling naturally — no interpolation required
- Theoretically principled (ODE-based continuous dynamics)
- Robust to missing values
- Captures long-range dependencies through continuous integration

**Limitations:**
- Computationally expensive (ODE solver at each forward pass)
- Complex implementation (requires `torchdiffeq` or `torchcde`)
- Requires careful numerical solver selection
- Limited community resources compared to GRU/TCN


In [ ]:
# ============================================================
# 10.6 Model Comparison Summary Table
# ============================================================

model_comparison = pd.DataFrame({
    'Model': ['Isolation Forest', 'One-Class SVM',
              'GRU Autoencoder', 'TCN Autoencoder', 'NCDE'],
    'Type': ['Ensemble Tree', 'Kernel SVM',
              'RNN Neural Net', 'CNN Neural Net', 'ODE Neural Net'],
    'Temporal Awareness': ['No', 'No', 'Yes (sequences)',
                            'Yes (dilated conv)', 'Yes (continuous)'],
    'Handles Irregular Sampling': ['No', 'No', 'No', 'No', 'Yes'],
    'Implementation Complexity': ['Low', 'Low', 'Medium', 'Medium', 'High'],
    'Recommended Stage': ['Stage 2 (baseline)', 'Stage 2 (baseline)',
                           'Stage 3', 'Stage 3', 'Stage 4 (advanced)'],
})

display(model_comparison)

---
# Section 11 — Final Conclusion

---

## 11.1 Summary of Stage 1 Achievements

This notebook has established a complete, **presentation-ready preprocessing pipeline** for spacecraft telemetry and telecommand anomaly detection. The following stages were completed:

---

### Dataset Understanding

| Dataset | Rows | Columns | Parameters/Commands | Time Span |
|---------|------|---------|--------------------|-----------|
| `telemetry_train.csv` | 5,000 | 3 | 15 parameters | ~13.9 hours |
| `telecommand_train.csv` | 1,000 | 3 | 10 command types | Same window |

Both datasets represent **healthy, nominal spacecraft operations** — the baseline our future anomaly detector will learn.

---

### Preprocessing Completed

| Step | Action | Outcome |
|------|--------|--------|
| Data Quality | Missing, duplicates, timestamp validation | All checks passed — clean data |
| Timestamp Parsing | String → datetime | Enabled temporal operations |
| Temporal Features | Extracted hour, minute, day, weekday, elapsed_sec | Time-awareness for models |
| Format Transformation | Long → Wide (pivot) | ML-ready feature matrix |
| NaN Handling | Forward-fill in wide format | No missing values remain |
| StandardScaler | Zero-mean, unit-variance | Ready for classical ML |
| MinMaxScaler | Bounded [0,1] output | Ready for neural networks |

---

### Features Generated

| Category | Features |
|----------|----------|
| **Raw** | `value` |
| **Temporal** | `hour`, `minute`, `day`, `weekday`, `elapsed_sec`, `minute_of_day`, `is_weekend` |
| **Rolling** | `rolling_mean_5`, `rolling_mean_10`, `rolling_std_5` |
| **Residual** | `deviation_from_mean` |
| **Rate** | `change_rate`, `abs_change_rate` |
| **Lag** | `lag_1`, `lag_2`, `lag_3` |
| **Statistical** | `z_score` |

---

### Dataset Ready for Anomaly Detection

The dataset is now ready for **Stage 2: Anomaly Injection & Model Development**. The preprocessing pipeline developed here will be applied to anomaly-injected test data without modification — ensuring a consistent, reproducible workflow.

**Recommended next steps:**
1. Inject synthetic anomalies (point anomalies, collective anomalies, contextual anomalies)
2. Apply Isolation Forest and One-Class SVM as baseline models
3. Progress to GRU Autoencoder for temporal anomaly detection
4. Evaluate using precision, recall, F1 on injected anomaly labels
5. Explore TCN Autoencoder and NCDE for advanced stages

---

> *This notebook was prepared as part of a spacecraft health monitoring research project.*  
> *All data is synthetic and generated to simulate realistic ISRO-class spacecraft telemetry.*


In [ ]:
# ============================================================
# 11.2 Save Final Processed Datasets
# ============================================================

import os
os.makedirs('processed', exist_ok=True)

# Long-format with all engineered features
telemetry_fe.to_csv('processed/telemetry_engineered.csv', index=False)

# Wide-format (ML-ready, raw values)
telemetry_wide.to_csv('processed/telemetry_wide.csv', index=False)

# Wide-format, StandardScaler applied
df_standard_out = pd.DataFrame(X_standard, columns=param_cols)
df_standard_out.insert(0, 'timestamp', telemetry_wide['timestamp'].values)
df_standard_out.to_csv('processed/telemetry_wide_standard_scaled.csv', index=False)

# Wide-format, MinMaxScaler applied
df_minmax_out = pd.DataFrame(X_minmax, columns=param_cols)
df_minmax_out.insert(0, 'timestamp', telemetry_wide['timestamp'].values)
df_minmax_out.to_csv('processed/telemetry_wide_minmax_scaled.csv', index=False)

# Processed telecommand
telecommand.to_csv('processed/telecommand_processed.csv', index=False)

print('Processed datasets saved to /processed/:')
for f in os.listdir('processed'):
    fpath = os.path.join('processed', f)
    size  = os.path.getsize(fpath)
    print(f'  {f:<45} ({size:,} bytes)')

print('\n[STAGE 1 COMPLETE] Dataset is ready for anomaly model development.')